In [1]:
#this is for Ver7.5's 2-stage test where the 1st stage is trained on synthetic aicc data and the 2nd stage is trained on train.csv, so the code is a bit different...
# Install packages
!pip install -q transformers sentencepiece
import warnings
warnings.filterwarnings('ignore')
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd
import torch
from tqdm import tqdm

print("="*60)
print("AKKADIAN TRANSLATION - NLLB INFERENCE")
print("="*60)

# Configuration
MAX_LENGTH = 256
MODEL_PATH = "/kaggle/input/datasets/jennyli13/ver-7-5-nllb-600m-model-two-stage-process/ver7.5_final-nllb-model"

# Load model
print("\n[1/3] Loading fine-tuned NLLB model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"Device: {device}")
print("✅ Model loaded successfully!")

# Load test data
print("\n[2/3] Loading test data...")
test_df = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/test.csv')
print(f"Test samples: {len(test_df)}")

# Translate function
def translate_batch(texts, batch_size=16):
    translations = []
    
    tokenizer.src_lang = "eng_Latn"
    target_lang_id = tokenizer.convert_tokens_to_ids("eng_Latn")
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", max_length=MAX_LENGTH, 
                          truncation=True, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=target_lang_id,
                max_length=MAX_LENGTH,
                num_beams=10,
                length_penalty=1.2,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )
        
        translations.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    
    return translations

# Generate predictions
print("\n[3/3] Generating predictions...")
predictions = translate_batch(test_df['transliteration'].tolist())

# Create submission
submission_df = pd.DataFrame({
    'id': test_df['id'], 
    'translation': predictions
})
submission_df.to_csv('submission.csv', index=False)

print("\n" + "="*60)
print("✅ DONE! submission.csv created")
print("="*60)
print(f"Rows: {len(submission_df)}")
print("\nFirst 3 predictions:")
print(submission_df.head(3))

AKKADIAN TRANSLATION - NLLB INFERENCE

[1/3] Loading fine-tuned NLLB model...


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Device: cuda
✅ Model loaded successfully!

[2/3] Loading test data...
Test samples: 4

[3/3] Generating predictions...


Translating: 100%|██████████| 1/1 [00:04<00:00,  4.19s/it]


✅ DONE! submission.csv created
Rows: 4

First 3 predictions:
   id                                        translation
0   0  From the Kanesh colony, and to Akil: The matte...
1   1  From the beginning of the City until the end o...
2   2  Like our representatives you spoke to us. Whet...
